**E-commerce Product Q&A Chatbot**. In the real world, product web pages contain crucial information split across text (specifications, descriptions) and images (infographics showing dimensions, ports, or features).

This script will scrape a given URL, extract the text, download the images, use **LLaVA** to summarize the images, and then index everything into a local ChromaDB so **Llama 3** can answer user questions.

### The Architecture

1. **Web Crawler:** `BeautifulSoup` to extract clean text and download images from `<img>` tags.
2. **Vision Model:** Ollama (`llava`) converts images into text-based metadata/summaries.
3. **Embedding:** Ollama (`nomic-embed-text`) embeds the scraped text and the image summaries.
4. **LLM Generation:** Ollama (`llama3`) answers user questions based on the retrieved context.

### Step 1: Install Dependencies

In [1]:
# Step 1: Install system compression tools and dependencies
!sudo apt-get update && sudo apt-get install -y zstd

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,764 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,026 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,307 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,221 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 http://

In [2]:
# Step 2: Download and install the core Ollama environment binary
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
import os, subprocess, time

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'

# Open a log file to catch the output so it doesn't fill the OS pipe
log_file = open('/content/ollama_server.log', 'w')

subprocess.Popen(
    ["ollama", "serve"],
    stdout=log_file,
    stderr=log_file
)

time.sleep(5)

### Environment Setup & Pulling the Vision Model

To process images, we need to add a Multimodal LLM to your Ollama setup. We will use **LLaVA** (Large Language-and-Vision Assistant). We also need the `unstructured` library to parse PDFs into separate text, table, and image elements.

In [4]:
# 1. Install required libraries for PDF parsing and Multimodal RAG
!pip install -qU unstructured[pdf] pdf2image poppler-utils
!pip install -qU langchain-chroma langchain-core

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 61.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 10.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 8.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.1/543.1 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 138.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 90.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 108.9 MB/s eta 0:00:00
 

In [5]:
# 2. Pull the Vision Model via Ollama
!ollama pull llava
!ollama pull llama3
!ollama pull nomic-embed-text

In [6]:
!pip install -qU requests beautifulsoup4 langchain langchain-chroma langchain-core sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.0/133.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.4/596.4 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 9.1 MB/s eta 0:00:00


### Step 2: The Web Scraper (Text & Images)

This production-style function extracts the text while simultaneously finding all images, resolving their URLs, and downloading them locally.

In [7]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [8]:
def scrape_website_multimodal(url, image_dir="scraped_images"):
    """Scrapes text and downloads images from a target URL."""
    if not os.path.exists(image_dir):
        os.makedirs(image_dir)

    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    response = requests.get(url, headers=headers)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'html.parser')

    # 1. Extract clean text (removing scripts and styles)
    for script in soup(["script", "style", "nav", "footer"]):
        script.extract()
    text_content = soup.get_text(separator="\n", strip=True)

    # 2. Extract and download images
    downloaded_images = []
    img_tags = soup.find_all('img')

    for i, img in enumerate(img_tags):
        img_url = img.get('src')
        if not img_url:
            continue

        # Handle relative URLs
        img_url = urljoin(url, img_url)

        try:
            img_data = requests.get(img_url, headers=headers).content
            img_ext = img_url.split('.')[-1].split('?')[0]
            if img_ext.lower() not in ['jpg', 'jpeg', 'png']:
                img_ext = 'jpg' # default fallback

            img_path = os.path.join(image_dir, f"image_{i}.{img_ext}")
            with open(img_path, 'wb') as handler:
                handler.write(img_data)
            downloaded_images.append(img_path)
        except Exception as e:
            print(f"Failed to download {img_url}: {e}")

    return text_content, downloaded_images

In [9]:
# --- Execution ---
target_url = "https://www.amazon.in/Wave-Smartwatch-Animated-Functional-Multiple/dp/B0FLF38P8G/ref=bmx_dp_d_sccl_1_5/261-8356948-7051947?pd_rd_w=opciE&content-id=amzn1.sym.a5646ec7-a2de-49f1-8d39-15f8f2eef501&pf_rd_p=a5646ec7-a2de-49f1-8d39-15f8f2eef501&pf_rd_r=9XPRZ98F5AGPZDRTDTQB&pd_rd_wg=4lyF4&pd_rd_r=ec91a420-abb0-4619-8595-06934a90fa98&pd_rd_i=B0FLF38P8G&th=1" # Replace with a any URL
print(f"Scraping {target_url}...")
website_text, image_paths = scrape_website_multimodal(target_url)
print(f"Scraped {len(website_text)} characters of text and {len(image_paths)} images.")

Scraping https://www.amazon.in/Wave-Smartwatch-Animated-Functional-Multiple/dp/B0FLF38P8G/ref=bmx_dp_d_sccl_1_5/261-8356948-7051947?pd_rd_w=opciE&content-id=amzn1.sym.a5646ec7-a2de-49f1-8d39-15f8f2eef501&pf_rd_p=a5646ec7-a2de-49f1-8d39-15f8f2eef501&pf_rd_r=9XPRZ98F5AGPZDRTDTQB&pd_rd_wg=4lyF4&pd_rd_r=ec91a420-abb0-4619-8595-06934a90fa98&pd_rd_i=B0FLF38P8G&th=1...
Scraped 17693 characters of text and 40 images.


### Step 3: Vision Processing with LLaVA

Websites often have tracking pixels, icons, or irrelevant logos. We use LLaVA to look at the downloaded images. If the image contains useful product info, LLaVA summarizes it.

In [10]:
!pip install -qU langchain-ollama

In [11]:
import os
import base64
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

In [12]:
# Initialize Vision Model
llava_chat = ChatOllama(model="llava", base_url="http://localhost:11434", temperature=0.1)

In [13]:
def generate_image_context(image_paths):
    """Passes images to LLaVA to generate descriptive text summaries."""
    image_summaries = []

    prompt = """You are an AI analyzing images from a website.
    Describe the contents of this image in detail. If it is a product, describe its features, colors, text, and dimensions shown.
    If it is a meaningless icon, logo, or blank image, reply with 'IRRELEVANT'."""

    for img_path in image_paths:
        # 1. Skip tiny files (less than 1KB are usually tracking pixels or tiny UI icons)
        if os.path.getsize(img_path) < 1024:
            print(f"Skipping {img_path} (file too small/likely an icon)...")
            continue

        with open(img_path, "rb") as image_file:
            encoded_string = base64.b64encode(image_file.read()).decode('utf-8')

        message = HumanMessage(
            content=[
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{encoded_string}"}}
            ]
        )

        print(f"Analyzing {img_path}...")

        # 2. Catch Ollama decoding errors so the loop doesn't crash
        try:
            response = llava_chat.invoke([message]).content

            # Filter out UI icons or tracking pixels
            if "IRRELEVANT" not in response.upper():
                # Tag the summary so the LLM knows it came from an image
                image_summaries.append(f"[IMAGE DESCRIPTION]: {response}")

        except Exception as e:
            print(f" --> Failed to process {img_path}, skipping. Error: {e}")
            continue

    return image_summaries

In [14]:
# --- Execution ---
print("Generating image summaries using LLaVA...")
image_contexts = generate_image_context(image_paths)

Generating image summaries using LLaVA...
Skipping scraped_images/image_0.jpg (file too small/likely an icon)...
Analyzing scraped_images/image_1.png...
Analyzing scraped_images/image_2.jpg...
Analyzing scraped_images/image_3.png...
Analyzing scraped_images/image_4.jpg...
Skipping scraped_images/image_5.jpg (file too small/likely an icon)...
Analyzing scraped_images/image_6.png...
Skipping scraped_images/image_7.jpg (file too small/likely an icon)...
Skipping scraped_images/image_8.jpg (file too small/likely an icon)...
Skipping scraped_images/image_9.jpg (file too small/likely an icon)...
Skipping scraped_images/image_10.jpg (file too small/likely an icon)...
Skipping scraped_images/image_11.jpg (file too small/likely an icon)...
Analyzing scraped_images/image_12.jpg...
Skipping scraped_images/image_13.jpg (file too small/likely an icon)...
Skipping scraped_images/image_14.jpg (file too small/likely an icon)...
Skipping scraped_images/image_15.jpg (file too small/likely an icon)...
An

### Step 4: Chunking & Vector Database Construction

We combine the text scraped from the HTML with the image descriptions generated by LLaVA, chunk them, and push them to ChromaDB.

In [18]:
!pip install -qU langchain-text-splitters langchain-community langchain-chroma

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.4 MB/s eta 0:00:00


In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.embeddings import OllamaEmbeddings
from langchain_chroma import Chroma

/tmp/ipykernel_809/2809949497.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import OllamaEmbeddings


In [20]:
# 1. Chunk the website text
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
text_chunks = text_splitter.split_text(website_text)

# 2. Create Document objects for both text and image summaries
documents = []
for chunk in text_chunks:
    documents.append(Document(page_content=chunk, metadata={"source": "website_text"}))

for img_context in image_contexts:
    documents.append(Document(page_content=img_context, metadata={"source": "website_image"}))

In [21]:
# 3. Embed into ChromaDB
embeddings = OllamaEmbeddings(model="nomic-embed-text", base_url="http://localhost:11434")

print("Embedding data into ChromaDB...")
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="./website_multimodal_db"
)

# Create a retriever that fetches the top 4 most relevant chunks
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

/tmp/ipykernel_809/2615376850.py:2: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text", base_url="http://localhost:11434")


Embedding data into ChromaDB...


### Step 5: The Q&A Chatbot Pipeline

Finally, we set up `llama3` to answer questions based on the retrieved text and image descriptions.


In [22]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [23]:
# Initialize the main text LLM
llama3_chat = ChatOllama(model="llama3", base_url="http://localhost:11434", temperature=0.2)

In [24]:
# Build the Chat Prompt
template = """You are a helpful customer support chatbot for a website.
Answer the user's question using ONLY the provided context.
The context includes text from the website as well as descriptions of images found on the page.

Context:
{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

In [25]:
def format_docs(docs):
    return "\n\n".join(f"Source: {doc.metadata['source']}\nContent: {doc.page_content}" for doc in docs)

In [26]:
# Create the RAG Chain
website_chatbot = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llama3_chat
    | StrOutputParser()
)

In [27]:
# --- Execution / Chat Loop ---
def chat_with_website(question):
    print(f"\nUser: {question}")
    response = website_chatbot.invoke(question)
    print(f"Chatbot: {response}")

In [28]:
# Test your production chatbot!
chat_with_website("What features are shown in the product images?")


User: What features are shown in the product images?
Chatbot: Based on the provided context, I can see that there is an image section with the following options:

* VIDEO
* 3+
* Image Unavailable
* IMAGES
* 360° VIEW

However, it seems that none of these options display any features of the product. The only text description available is for the "Boat Wave Call 3 Smartwatch" which mentions its specifications such as a 1.83” HD Display with Animated Watch Faces; BT Calling, Functional Crown, Multiple Sports Modes, IP68, HR, SpO2 Monitor.

Therefore, I would answer that there are no product features shown in the product images.


In [29]:
chat_with_website("What is the price and return policy?")


User: What is the price and return policy?
Chatbot: According to the provided context, the price of the device is ₹1,399.00. As for the return policy, it's mentioned that "Returns will not be accepted if it is an Open Box Delivery order" and that you can visit the nearest boAt Service Centre for product-related issues. Additionally, there is a 1 Year Warranty from the date of purchase.


In [30]:
chat_with_website("What colors is the product available in based on the photos?")


User: What colors is the product available in based on the photos?
Chatbot: Based on the context, there is only one color option mentioned for the product, which is "Spring Blossom". There are no other color options shown or mentioned.
